# Bioturbation model conceptualisation

The bioturbation model predicts how earthworm bioturbation effects concentration of a contaminant in a layered soil model, where a certain depth of each layer is mixed with the depth of other layer(s) on each time step. The depth to mix is a function of earthworm density in the layer. For more details, see [Baccaro M, Harrison S, et al 2019](https://doi.org/10.1016/j.envpol.2019.05.106).

The soil profile is split into a number $L$ of soil layers, each with a depth $d_l$ [m] (where $l \in \{ 1, ..., L \}$) and concentration of contaminant $\text{[A]}_l$ [kg/m<sup>3</sup>]. Each pair of layers $(i, j)$ has a transport rate $K_{ij}$ [s<sup>-1</sup>] that determines how rapidly mass is exchanged between them. We collect these rates into a symmetric matrix $K$ with zero diagonal.

![Conceptual model of the soil profile.](../img/conceptual-model.png)

On each time step of length $\delta t$, the concentration in layer $i$ is updated by summing the pairwise fluxes to every other layer, evaluated from the *old* state:

$$
    \text{[A]}_{i,t+1} = \text{[A]}_{i,t} + \delta t \sum_{j \neq i} K_{ij} \left( \text{[A]}_{j,t} - \text{[A]}_{i,t} \right)
$$

Total mass $d \sum_i \text{[A]}_i$ is conserved exactly when $K$ is symmetric (equal-depth case). For unequal depths the requirement generalises to $d_i K_{ij} = d_j K_{ji}$ — the current model assumes equal depths. The explicit time-stepping scheme is stable when $\delta t \cdot \max_i \sum_{j \neq i} K_{ij} \le 1$.

## Default $K$: adjacent-only mixing

When no transport matrix is supplied, the model builds a tri-diagonal $K$ from earthworm density and a calibration parameter:

$$
    K_{l,l+1} = K_{l+1,l} = \frac{m_{l:l+1}}{d_l}, \qquad m_{l:l+1} = \beta w_l
$$

where $w_l$ [worms/m<sup>3</sup>] is the earthworm density in the upper layer of the pair and $\beta$ [m<sup>4</sup> s<sup>-1</sup>] is the bioturbation calibration parameter, usually fitted to experimental data (see [calibrate.ipynb](./calibrate.ipynb)). All other entries of $K$ are zero, recovering classical adjacent-only mixing.

## Non-adjacent transport

Passing a user-defined `transport_matrix=K` to `run()` enables transport between any pair of layers — for example, representing deep-burrowing (anecic) earthworms that move tracers directly from surface to deep layers. See [non_adjacent.ipynb](./non_adjacent.ipynb) for a worked example.